# MVP2 FDTD 128x128 -> 2D contour -> HDMI (4-lane)
**128x128** grid (4x finer than 64x64) on the 4-lane FDTD + 2D contour heatmap. source_addr is now 14-bit. Use `mag_mode=2` (signed Ey).
AXI: `gpio_ctrl 0x41200000` | `gpio_motion 0x41210000` | `gpio_status 0x41220000`.


In [ ]:
from pynq import Overlay, MMIO
import time
ol = Overlay('fdtd_hdmi.bit')
print('overlay loaded'); print(list(ol.ip_dict.keys()))


In [ ]:
CTRL   = MMIO(0x41200000, 0x10000)
STATUS = MMIO(0x41220000, 0x10000)
MOTION = MMIO(0x41210000, 0x10000)   # moving source velocity + speed throttle
GPIO_CH1, GPIO_CH2 = 0x0, 0x8
GRID = 128
def cell(x, y): return y*GRID + x
def q313(v):    return int(round(v*8192)) & 0xFFFF

def set_ctrl(phase_step, amplitude, source_addr, solver_enable, mag_mode,
             sample_req, free_run, height_ctl=2):
    """mag_mode: 0=|E|, 1=|S|, 2=raw signed Ey (smooth waves). height_ctl signed -16..15."""
    ch1 = (q313(amplitude) << 16) | q313(phase_step)
    ch2 = (((mag_mode >> 1) & 1) << 23) | ((height_ctl & 0x1F) << 18) | ((free_run & 1) << 17) \
        | ((sample_req & 1) << 16) | ((mag_mode & 1) << 15) | ((solver_enable & 1) << 14) \
        | (source_addr & 0x3FFF)
    CTRL.write(GPIO_CH1, ch1); CTRL.write(GPIO_CH2, ch2)

def set_amplitude(amp):
    """Set source amplitude (Q3.13 float, e.g. 0..0.5) live; keeps phase_step."""
    ch1 = (CTRL.read(GPIO_CH1) & 0x0000FFFF) | (q313(amp) << 16)
    CTRL.write(GPIO_CH1, ch1)

def set_height(h):
    ch2 = (CTRL.read(GPIO_CH2) & ~(0x1F<<18)) | ((h & 0x1F)<<18); CTRL.write(GPIO_CH2, ch2)

def clear_fields():
    """Reset the simulation: zero Ey/Ex/Bz (all 4 lanes). Edge-triggered bit 24."""
    v = CTRL.read(GPIO_CH2)
    CTRL.write(GPIO_CH2, v | (1<<24)); time.sleep(0.005); CTRL.write(GPIO_CH2, v & ~(1<<24))
    print("fields cleared")

def read_status():
    chk = STATUS.read(GPIO_CH1); s = STATUS.read(GPIO_CH2)
    return dict(checksum=chk, solver_done=(s>>0)&1, source_valid=(s>>1)&1, mag_done=(s>>2)&1,
                mag_busy=(s>>3)&1, source_latched=(s>>4)&1, pp_read_sel=(s>>5)&1,
                pp_frame_ready=(s>>6)&1, bridge_busy=(s>>7)&1, source_q313=(s>>16)&0xFFFF)
print("helpers ready")

def _s16(v):  # float cells/iter -> signed 16-bit (x256 fixed point)
    iv = int(round(v*256))
    iv = max(-32768, min(32767, iv))
    return iv & 0xFFFF

def set_velocity(vx, vy):
    """Move the source at (vx,vy) cells per FDTD iteration. The velocity is
       latched on a move_en rising edge, so we pulse move_en low->high here to
       reload it live (the source restarts from the source cell each call).
       Subsonic |v| -> Doppler (rings compress ahead); supersonic -> Mach cone.
       The numerical wave speed is < 0.5 cell/iter -- sweep v upward to find the
       regimes. Call stop_motion() to halt."""
    MOTION.write(0x0, (_s16(vy) << 16) | _s16(vx))
    base = MOTION.read(0x8) & ~0x1          # move_en low (re-arm the edge)
    MOTION.write(0x8, base)
    MOTION.write(0x8, base | 0x1)           # rising edge -> reload vx,vy + restart
    print(f"moving source: vx={vx} vy={vy} cells/iter (sweep v up: rings->Doppler->Mach cone)")

def stop_motion():
    MOTION.write(0x8, MOTION.read(0x8) & ~0x1)
    print("source motion stopped (back to static source_addr)")

def set_speed(idle_cycles):
    """Throttle: idle cycles inserted between FDTD iterations. 0 = full speed;
       larger = slower (watchable). e.g. set_speed(200000) ~ visibly slowed."""
    ch2 = (MOTION.read(0x8) & 0xFF) | ((idle_cycles & 0xFFFFFF) << 8)
    MOTION.write(0x8, ch2)
    print(f"speed throttle: {idle_cycles} idle cycles/iteration")

def set_source_mode(dcfree=True):
    """dcfree=True: inject the DC-free first-difference of the sine. A MOVING
       source otherwise deposits a net DC offset on each cell it passes, which
       freezes into a static 'trail'; the difference telescopes -> no trail.
       Stationary waves are unchanged. If waves look weak, raise amplitude or
       set_height(+1..+2) for display gain."""
    ch2 = MOTION.read(0x8)
    ch2 = (ch2 | 0x2) if dcfree else (ch2 & ~0x2)
    MOTION.write(0x8, ch2)
    print(f"source mode: {'DC-free (no trail)' if dcfree else 'direct sine'}")

def set_source_field(field='Bz'):
    """Choose which field the point source drives (motion CH2 bit2):
       'Ey' = dipole source -> figure-8 pattern with two nulls (real EM dipole);
       'Bz' = monopole source -> isotropic CIRCULAR ripples (best for Doppler/Mach,
              no quiet wedges). Switchable live."""
    bz = 1 if str(field).lower().startswith('b') else 0
    ch2 = MOTION.read(0x8)
    ch2 = (ch2 | 0x4) if bz else (ch2 & ~0x4)
    MOTION.write(0x8, ch2)
    print(f"source field: {'Bz (monopole, circular)' if bz else 'Ey (dipole, two nulls)'}")


## 5. UDP source-magnitude control (ESP32 -> PS)
The ESP32 streams the probe value over WiFi/UDP (port **5005**, ASCII int per packet); a background thread on the PS maps it to the FDTD **source amplitude** live. Both devices must be on the same local network. ESP32 sketch: `esp32/source_magnitude_udp.ino`.

First calibrate the probe range with `mc.calibrate()` (move the probe through its full range, note min/max), then set `RAW_MIN/RAW_MAX`.

In [ ]:
import socket, threading, time

class MagnitudeUDP:
    """Receive probe values over UDP and drive the FDTD source amplitude.
    Packet = one ASCII integer per datagram (e.g. b'2731')."""
    def __init__(self, port=5005, raw_min=0, raw_max=4095, amp_min=0.0, amp_max=0.5):
        self.port=port; self.raw_min=raw_min; self.raw_max=raw_max
        self.amp_min=amp_min; self.amp_max=amp_max
        self.sock=socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        self.sock.bind(('0.0.0.0', port)); self.sock.setblocking(False)
        self._run=False; self._t=None; self.last_raw=None; self.last_amp=None
    def _latest(self):
        data=None
        while True:
            try: data,_=self.sock.recvfrom(64)
            except BlockingIOError: break
        if data is None: return None
        try: return int(data.decode('ascii','ignore').strip().split(',')[0])
        except (ValueError, IndexError): return None
    def _to_amp(self, raw):
        f=(raw-self.raw_min)/max(1,(self.raw_max-self.raw_min)); f=min(1.0,max(0.0,f))
        return self.amp_min + f*(self.amp_max-self.amp_min)
    def _loop(self):
        while self._run:
            r=self._latest()
            if r is not None:
                self.last_raw=r; self.last_amp=self._to_amp(r); set_amplitude(self.last_amp)
            time.sleep(0.01)
    def start(self):
        if not self._run:
            self._run=True; self._t=threading.Thread(target=self._loop, daemon=True); self._t.start()
            print(f'UDP magnitude control live on :{self.port}')
    def stop(self):
        self._run=False; time.sleep(0.05); print('stopped')
    def calibrate(self, secs=8):
        """Move the probe through its full range; prints min/max raw seen."""
        lo,hi=1<<30,-(1<<30); t0=time.time()
        print('move the probe through its full range...')
        while time.time()-t0 < secs:
            r=self._latest()
            if r is not None: lo=min(lo,r); hi=max(hi,r)
            time.sleep(0.01)
        print(f'raw range: min={lo} max={hi}  -> set raw_min/raw_max to these')
        return lo,hi

# start it (source stays at the centre; its amplitude tracks the probe)
mc = MagnitudeUDP(port=5005, raw_min=0, raw_max=4095, amp_min=0.0, amp_max=0.5)
# mc.calibrate()          # run once, then set raw_min/raw_max from the printout
mc.start()
# mc.stop()               # to stop the background thread / re-run the cell


## 1. Start free-run (4-lane solver)


In [ ]:
# NOTE: phase now advances phase_step PER ITERATION (CORDIC re-timed to
# one sample/iter). ~0.18 rad => ~12-cell wavelength; bigger=shorter waves.
# source at y=48 = centre of a lane (seams at y=32/64/96); keeps rings
# symmetric. y=64 is ON a seam -> top/bottom look mismatched.
set_ctrl(phase_step=0.18, amplitude=0.3, source_addr=cell(64,48),
         solver_enable=1, mag_mode=2, sample_req=1, free_run=1, height_ctl=2)
print("4-lane FDTD running. Tune relief with set_height(n); reset with clear_fields().")


## Doppler / moving source
Move the source through the grid. With `mag_mode=2` you'll see wavefronts bunch up ahead (higher freq) and stretch behind (lower freq). Push the speed past the wave speed (~0.35 cells/iter) for a Mach cone. Throttle the sim first so it's watchable.

In [ ]:
set_source_mode(True)    # DC-free injection: no trail
set_source_field('Bz')   # monopole -> full circular wavefronts (no two-sided nulls)
set_speed(150000)        # slow the sim so the motion is watchable

# velocity now applies LIVE (no notebook restart needed). Sweep it to see the
# regimes -- the numerical wave speed is somewhere < 0.5 cell/iter:
set_velocity(0.08, 0.0)  #  slow  -> near-symmetric rings (little Doppler)
# set_velocity(0.15, 0.0)  # medium -> Doppler: rings bunch ahead, stretch behind
# set_velocity(0.30, 0.0)  # near/above wave speed -> fronts pile into a cone
# set_velocity(0.45, 0.0)  # clearly supersonic -> Mach cone / shock
# set_velocity(0.18, 0.10) # diagonal
# stop_motion()            # halt (source stays put)


## 2. Confirm alive
Checksum must keep changing and ping-pong must swap. The solver now produces frames ~4x faster than the single-lane build.


In [ ]:
seen=set(); flips=0; prev=None
for _ in range(20):
    st=read_status(); seen.add(st['checksum'])
    if prev is not None and st['pp_read_sel']!=prev: flips+=1
    prev=st['pp_read_sel']; time.sleep(0.05)
print("unique checksums:", len(seen), " read_sel flips:", flips)
assert len(seen)>1, "checksum frozen"
print("PASS: 4-lane solver live.")


## 3. Reset / clean the renderer


In [ ]:
clear_fields()   # zero the field, wave restarts from the source


## 4. HDMI
Expect the **same** 3D wave terrain as the single-lane build (try `mag_mode=2` signed-Ey). The win is internal: the solver advances 4x faster per iteration. Tuning: `set_height(1..4)`, `phase_step` for wavelength (now per-iteration: ~0.18 rad ≈ 12 cells; raise for shorter waves), `mag_mode` 0/1/2 for view.


## 6. Camera control (live 3D view)

The ray-march camera basis is now PS-controllable (GPIO `cam_a/b/c` at `0x41230000/40000/50000`, latched by `cam_load` = MOTION CH2 bit3). The PS does the trig; the FPGA consumes the 12 Q3.13 basis vectors. `set_camera()` orbits/tilts/zooms the view with **no rebuild**. Defaults reproduce the original view's angle (eye recentred on the grid). The board boots to the exact hard-wired view, so `set_camera()` is only needed to *move* the view.

In [ ]:
import math
CAMA = MMIO(0x41230000, 0x10000)
CAMB = MMIO(0x41240000, 0x10000)
CAMC = MMIO(0x41250000, 0x10000)

def _q(v):  # world units (1.0 = 8192) -> signed 16-bit two's complement
    iv = int(round(v*8192)); iv = max(-32768, min(32767, iv)); return iv & 0xFFFF

def set_camera(yaw_deg=45.0, pitch_deg=45.0, dist=0.64):
    """Orbit the 3D camera around the heightmap centre. The grid spans world
       [-1,+1] (1.0 = 8192); terrain base at z=0.
         yaw_deg   : azimuth about the vertical axis
         pitch_deg : look-down angle, 0=horizon .. 90=top-down (clamped 2..89)
         dist      : camera distance from centre, world units (~0.4..2.0;
                     >3 overflows Q3.13 -> clamps).
       Defaults match the hard-wired view's ANGLE exactly; the eye is recentred
       on the grid (~3% origin shift vs the boot view). The PS computes the
       orthonormal basis; the FPGA just consumes it, latched on a cam_load pulse."""
    pitch_deg = max(2.0, min(89.0, pitch_deg))
    ya, pa = math.radians(yaw_deg), math.radians(pitch_deg)
    cy, sy, cp, sp = math.cos(ya), math.sin(ya), math.cos(pa), math.sin(pa)
    fwd = (cp*cy, cp*sy, -sp)                      # camera -> target, tilted down
    O   = (-dist*fwd[0], -dist*fwd[1], -dist*fwd[2])  # target = (0,0,0)
    rgt = (fwd[1], -fwd[0], 0.0)                   # fwd x world_up
    rn  = math.hypot(rgt[0], rgt[1]) or 1.0
    rgt = (rgt[0]/rn, rgt[1]/rn, 0.0)
    up  = (rgt[1]*fwd[2]-rgt[2]*fwd[1],            # right x fwd (orthonormal)
           rgt[2]*fwd[0]-rgt[0]*fwd[2],
           rgt[0]*fwd[1]-rgt[1]*fwd[0])
    CAMA.write(0x0, (_q(O[1])  <<16)|_q(O[0]))     # {oy, ox}
    CAMA.write(0x8, (_q(fwd[0])<<16)|_q(O[2]))     # {fwd_x, oz}
    CAMB.write(0x0, (_q(fwd[2])<<16)|_q(fwd[1]))   # {fwd_z, fwd_y}
    CAMB.write(0x8, (_q(rgt[1])<<16)|_q(rgt[0]))   # {right_y, right_x}
    CAMC.write(0x0, (_q(up[0]) <<16)|_q(rgt[2]))   # {up_x, right_z}
    CAMC.write(0x8, (_q(up[2]) <<16)|_q(up[1]))    # {up_z, up_y}
    base = MOTION.read(0x8) & ~(1<<3)              # latch: cam_load low->high
    MOTION.write(0x8, base); MOTION.write(0x8, base | (1<<3))
    print(f'camera: yaw={yaw_deg} pitch={pitch_deg} dist={dist}  (latched)')

def reset_camera(): set_camera(45.0, 45.0, 0.64)   # back to default isometric

# The board already boots to the exact hard-wired isometric view (RTL default),
# so you do NOT need to call set_camera() to see the scene -- only to move it.
# reset_camera()                   # snap to the centred isometric view
# Try:  set_camera(yaw_deg=0)       # face along +x
#       set_camera(pitch_deg=80)    # near top-down
#       set_camera(dist=1.2)        # zoom out
#       for a in range(0,360,15): set_camera(yaw_deg=a); time.sleep(0.1)  # spin


## 7. Hardware panel input (ESP32 resistive touch panel -> PS over USB)

The ESP32 one-panel firmware streams readable `DATA,...` records over USB serial at about 50 Hz. `PanelReceiver` parses those records and reuses the helper functions from earlier cells where available, writing only the missing source-position register directly:

- touch `x,y` -> `CTRL` CH2 `source_addr[13:0]` (`require_mode2=False` by default)
- `amp` -> `CTRL` CH1 `amplitude_q313[31:16]`
- `clear` rising edge -> previous `clear_fields()` helper
- `field` -> previous `set_source_field()` helper for E/B source selection
- `yaw,pitch,zoom` -> camera GPIOs only when `PanelReceiver(do_camera=True)` is used

The ESP32 panel reports `x,y` as 128x128 grid cells: `0..127` when touched, and `-1,-1` when not touched. The notebook maps those into the `GRID x GRID` FDTD address space; with `GRID=128` this is a direct 1:1 mapping. This bitstream has no AXI wall-memory interface; `PanelReceiver` therefore ignores the ESP32 wall switch by default and still uses touched coordinates for source placement. Port is usually `/dev/ttyUSB0` or `/dev/ttyACM0`.


Use `test_source_positions()` first to verify AXI writes, then use `pr.run(...)` or repeated `pr.poll()` for ESP32-driven AXI writes. `pr.start()` is read-only by default to avoid PYNQ notebook kernel crashes from background-thread MMIO.


In [ ]:
import glob, threading, time
try:
    import serial
    from serial.tools import list_ports
except ImportError:
    serial = None
    list_ports = None
    print('pyserial not installed in this notebook kernel. Run: import sys; !{sys.executable} -m pip install pyserial')

# ESP32 firmware reports 128x128 panel grid cells x=0..127, y=0..127.
PANEL_GRID_X, PANEL_GRID_Y = 128, 128
ADDR_MASK = 0x3FFF

# CTRL CH2 bit layout from scripts/create_fdtd_render_project.tcl:
# [13:0] source_addr, [14] solver_enable, [15] mag_mode[0], [16] sample_req,
# [17] free_run, [22:18] height_ctl, [23] mag_mode[1], [24] clear_req.
BIT_SOLVER_ENABLE = 14
BIT_MAG_MODE_LO   = 15
BIT_SAMPLE_REQ    = 16
BIT_FREE_RUN      = 17
BIT_HEIGHT_CTL    = 18
BIT_MAG_MODE_HI   = 23
BIT_CLEAR_REQ     = 24

# MOTION CH2: [0] move_en, [1] src_dcfree, [2] source_bz, [3] cam_load, [31:8] speed_div.
BIT_MOVE_EN    = 0
BIT_SRC_DCFREE = 1
BIT_SOURCE_BZ  = 2
BIT_CAM_LOAD   = 3


def find_panel_port():
    """Return the first likely ESP32 USB serial device."""
    candidates = sorted(glob.glob('/dev/ttyUSB*') + glob.glob('/dev/ttyACM*'))
    if candidates:
        return candidates[0]
    if list_ports is not None:
        ports = list(list_ports.comports())
        for p in ports:
            text = f'{p.device} {p.description} {p.hwid}'.lower()
            if any(name in text for name in ('cp210', 'ch340', 'wch', 'silicon labs', 'esp32', 'usb serial')):
                return p.device
        if ports:
            return ports[0].device
    raise RuntimeError('No ESP32 serial port found. Check: ls /dev/ttyUSB* /dev/ttyACM*')


def _clamp_int(v, lo, hi):
    return max(lo, min(hi, int(v)))


def set_source_position(x, y):
    """Write CTRL CH2 source_addr without disturbing solver/mag/free-run/height bits."""
    x = _clamp_int(round(x), 0, GRID - 1)
    y = _clamp_int(round(y), 0, GRID - 1)
    # Static stylus placement owns source_addr, so disable the moving-source engine.
    MOTION.write(GPIO_CH2, MOTION.read(GPIO_CH2) & ~(1 << BIT_MOVE_EN))
    v = CTRL.read(GPIO_CH2)
    CTRL.write(GPIO_CH2, (v & ~ADDR_MASK) | (cell(x, y) & ADDR_MASK))


def get_source_position():
    """Read back CTRL CH2 source_addr as (x, y, addr)."""
    addr = CTRL.read(GPIO_CH2) & ADDR_MASK
    return addr & (GRID - 1), (addr >> 7) & (GRID - 1), addr


def panel_is_touched(d):
    """True only when the firmware reports a valid panel coordinate."""
    if 'x' not in d or 'y' not in d:
        return False
    if int(d['x']) < 0 or int(d['y']) < 0:
        return False
    if 'touch' in d and int(d.get('touch', 0) or 0) == 0:
        return False
    return True


def _panel_to_fdtd(px, py):
    """Scale ESP32 128x128 panel grid cells to the current FDTD GRID x GRID cell."""
    px = _clamp_int(px, 0, PANEL_GRID_X - 1)
    py = _clamp_int(py, 0, PANEL_GRID_Y - 1)
    fx = round(px * (GRID - 1) / (PANEL_GRID_X - 1))
    fy = round(py * (GRID - 1) / (PANEL_GRID_Y - 1))
    return fx, fy


def parse_data_line(s):
    """Parse one ESP32 'DATA,k=v,...' line. Boot/calibration/debug lines return None."""
    if isinstance(s, bytes):
        s = s.decode('ascii', 'ignore')
    s = s.strip()
    if not s.startswith('DATA,'):
        return None
    d = {}
    for item in s[5:].split(','):
        if '=' not in item:
            continue
        key, value = item.split('=', 1)
        key = key.strip().lower()
        value = value.strip()
        try:
            d[key] = int(value, 0)
        except ValueError:
            d[key] = value
    return d


class PanelReceiver:
    """ESP32 USB DATA line -> AXI register mapper for the D5S128 FDTD bitstream.

    Use poll()/run() for AXI writes. start() is read-only by default because
    repeated MMIO writes from a Jupyter background thread can crash the PYNQ
    kernel instead of raising a Python exception.
    """
    def __init__(self, port=None, baud=115200,
                 do_position=True, do_amp=True, do_clear=True,
                 do_field=True, do_camera=False, debug=False, ignore_wall=True,
                 require_mode2=False, amp_max=0.50, amp_deadband=8, camera_period=0.12):
        if serial is None:
            raise RuntimeError('pyserial is not installed in this notebook kernel')
        self.port = port or find_panel_port()
        self.ser = serial.Serial(self.port, baud, timeout=0)
        self.ser.reset_input_buffer()
        self.do_position = do_position
        self.do_amp = do_amp
        self.do_clear = do_clear
        self.do_field = do_field
        self.do_camera = do_camera and ('set_camera' in globals())
        self.debug = debug
        self.ignore_wall = ignore_wall
        self.require_mode2 = require_mode2
        self.amp_max = float(amp_max)
        self.amp_deadband = int(amp_deadband)
        self.camera_period = float(camera_period)
        self._run = False
        self._t = None
        self._apply_in_thread = False
        self._buf = b''
        self.last = None
        self.last_fdtd_xy = None
        self._clear_prev = 0
        self._amp_prev = None
        self._field_prev = None
        self._cam_prev = None
        self._cam_t = 0.0
        self._debug_t = 0.0

    def _read_lines(self):
        try:
            chunk = self.ser.read(4096)
        except Exception as e:
            print('serial read failed:', e)
            return []
        if chunk:
            self._buf += chunk
        if b'\n' not in self._buf:
            return []
        parts = self._buf.split(b'\n')
        self._buf = parts[-1]
        return [p.decode('ascii', 'ignore').strip() for p in parts[:-1]]

    def _read_records(self):
        records = []
        for line in self._read_lines():
            rec = parse_data_line(line)
            if rec is not None:
                self.last = rec
                records.append(rec)
        return records

    def read_one(self, timeout=2.0):
        """Blocking single-frame read, useful before doing any AXI writes."""
        end = time.time() + timeout
        while time.time() < end:
            records = self._read_records()
            if records:
                return records[-1]
            time.sleep(0.005)
        return None

    def _apply_field(self, field):
        # Reuse helpers defined in earlier cells. ESP32: 0=E, 1=B, 2=S.
        if field == self._field_prev:
            return
        self._field_prev = field
        if field == 0 and 'set_source_field' in globals():
            set_source_field('Ey')
        elif field == 1 and 'set_source_field' in globals():
            set_source_field('Bz')
        # field==2 is display/magnitude selection; leave the current mag_mode alone.

    def _apply(self, d):
        if self.do_clear:
            clear_now = int(d.get('clear', 0) or 0)
            if clear_now and not self._clear_prev and 'clear_fields' in globals():
                clear_fields()
            self._clear_prev = clear_now

        if self.do_field and 'field' in d:
            self._apply_field(int(d['field']))

        position_mode_ok = (not self.require_mode2) or d.get('mode') == 2
        if self.do_position and position_mode_ok and panel_is_touched(d):
            # This bitstream has no AXI wall RAM/control path. By default, still use
            # touched coordinates for source placement even if the ESP32 wall switch is set.
            if self.ignore_wall or int(d.get('wall', 0) or 0) == 0:
                fx, fy = _panel_to_fdtd(d['x'], d['y'])
                set_source_position(fx, fy)
                self.last_fdtd_xy = (fx, fy)
                if self.debug and time.time() - self._debug_t >= 0.10:
                    self._debug_t = time.time()
                    rb_x, rb_y, rb_addr = get_source_position()
                    print(f'panel ({d["x"]},{d["y"]}) -> fdtd ({fx},{fy}) addr={cell(fx, fy)} readback=({rb_x},{rb_y})/0x{rb_addr:04x}')

        if self.do_amp and 'amp' in d:
            amp_raw = _clamp_int(d['amp'], 0, 1023)
            if self._amp_prev is None or abs(amp_raw - self._amp_prev) >= self.amp_deadband:
                self._amp_prev = amp_raw
                if 'set_amplitude' in globals():
                    set_amplitude((amp_raw / 1023.0) * self.amp_max)

        if self.do_camera and all(k in d for k in ('yaw', 'pitch', 'zoom')):
            cur = (int(d['yaw']), int(d['pitch']), int(d['zoom']))
            now = time.time()
            changed = self._cam_prev is None or any(abs(a - b) >= 12 for a, b in zip(cur, self._cam_prev))
            if changed and now - self._cam_t >= self.camera_period:
                self._cam_prev = cur
                self._cam_t = now
                set_camera(yaw_deg=cur[0] / 1023.0 * 360.0,
                           pitch_deg=2.0 + cur[1] / 1023.0 * 87.0,
                           dist=0.4 + cur[2] / 1023.0 * 1.6)

    def _latest_position_record(self, records):
        for rec in reversed(records):
            if ((not self.require_mode2) or rec.get('mode') == 2) and panel_is_touched(rec):
                if self.ignore_wall or int(rec.get('wall', 0) or 0) == 0:
                    return rec
        return None

    def poll(self, apply=True, latest_only=True):
        """Read available serial records once. If apply=True, write AXI in foreground."""
        records = self._read_records()
        if apply and records:
            if latest_only:
                # Use the newest valid touch-position record, not blindly the final
                # serial line. The final line is often a no-touch frame after jitter.
                pos = self._latest_position_record(records)
                self._apply(pos if pos is not None else records[-1])
            else:
                for rec in records:
                    self._apply(rec)
        return records[-1] if records else None

    def run(self, seconds=None, period=0.02):
        """Foreground loop: reads ESP32 and writes AXI. Interrupt the cell to stop."""
        t0 = time.time()
        while seconds is None or time.time() - t0 < seconds:
            self.poll(apply=True, latest_only=True)
            time.sleep(period)

    def _loop(self):
        while self._run:
            records = self._read_records()
            if self._apply_in_thread and records:
                self._apply(records[-1])
            time.sleep(0.01)

    def start(self, apply=False):
        """Start background serial reading. Default is read-only to protect PYNQ MMIO."""
        if self._run:
            return
        self._apply_in_thread = bool(apply)
        self._run = True
        self._t = threading.Thread(target=self._loop, daemon=True)
        self._t.start()
        mode = 'AXI-writing background mode' if apply else 'read-only background mode'
        print(f'panel receiver live on {self.port}: {mode}')
        if apply:
            print('warning: background MMIO can crash some PYNQ notebook kernels; prefer pr.run(...)')

    def stop(self):
        self._run = False
        if self._t is not None:
            self._t.join(timeout=0.2)
        print('panel receiver stopped')

    def close(self):
        self.stop()
        self.ser.close()


def test_source_positions(delay=1.0):
    """Manual AXI test independent of ESP32 serial."""
    for x, y in [(10, 10), (64, 64), (110, 90)]:
        set_source_position(x, y)
        print('wrote', (x, y), 'readback', get_source_position())
        time.sleep(delay)


# --- safe usage on PYNQ ---
# test_source_positions()                # first verify AXI source_addr writes without ESP32
# pr = PanelReceiver(do_camera=False, debug=True)  # mode switch is ignored for source position by default
# print(pr.read_one(timeout=2.0))         # verify parsed DATA line before AXI writes
# pr.poll()                              # one foreground read + AXI update; prints readback when touched
# pr.run(seconds=30)                     # foreground continuous mapping; interrupt cell to stop
#
# pr.start()                             # read-only background monitor; does NOT write AXI
# print(pr.last, pr.last_fdtd_xy)
# pr.stop()
#
# Avoid pr.start(apply=True) unless you have verified background MMIO is stable on your image.
